# Deep Learning — NLP & Transformers Part 1
## Classic Embeddings to Sentence Transformers

> Every concept in three layers: **Intuition → Math → Code**

---

## Table of Contents

| Section | Topic |
|---------|-------|
| **1** | The Problem with One-Hot Encoding |
| **2** | Word2Vec: CBOW & Skip-Gram |
| **3** | The Softmax Bottleneck & Negative Sampling |
| **4** | GloVe: Global Vectors & Matrix Factorization |
| **5** | FastText: Subword Information & OOV Words |
| **6** | Static vs. Contextual Embeddings (The Polysemy Problem) |
| **7** | Sentence Transformers (SBERT) & Siamese Networks |
| **8** | PyTorch Implementation Demo (Skip-Gram) |
| **9** | Master Interview Q&A Cheatsheet |


In [1]:
# ── All imports ────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
    "axes.prop_cycle": plt.cycler(color=[
        "#4C72B0","#DD8452","#55A868","#C44E52","#8172B3","#937860"
    ])
})
print("Imports ready ✓")


## 1. The Problem with One-Hot Encoding

Before deep learning revolutionized NLP, computers processed text by counting words. The foundational representation was **One-Hot Encoding**.

### The Intuition
Imagine a massive filing cabinet with 100,000 drawers (our vocabulary). If we want to represent the word "king", we open the drawer labeled "king" and put a '1' inside, leaving all other 99,999 drawers with a '0'.

**The Fatal Flaw:** In this system, every single word is an independent drawer. The drawer for "king" is completely disconnected from the drawer for "queen" or "man". The computer has zero concept that these words are related.

### The Math
Let vocabulary $V = \{\text{king}, \text{queen}, \text{man}, \text{apple}\}$.
Size $|V| = 4$.
$$ v_{\text{king}} = [1, 0, 0, 0] $$
$$ v_{\text{queen}} = [0, 1, 0, 0] $$

To measure similarity between vectors, we use the dot product (or cosine similarity). For any two unique words in a one-hot representation:
$$ v_{\text{king}} \cdot v_{\text{queen}} = (1 \times 0) + (0 \times 1) + (0 \times 0) + (0 \times 0) = 0 $$
**Result:** Mathematical Orthogonality. The distance between "king" and "queen" is exactly the same as the distance between "king" and "apple". Semantic relationships are utterly lost.

**Additionally:** Curse of Dimensionality. If $|V| = 100,000$, every word is a vector of $100,000$ elements, 99.999% of which are zeros. It is catastrophically inefficient memory-wise.

---

## 2. Word2Vec: CBOW & Skip-Gram

In 2013, researchers at Google (Mikolov et al.) proposed a breakthrough: **Dense Embeddings via Word2Vec**. Instead of huge, sparse vectors, what if we represented words as short, dense vectors (e.g., length 300) where the numbers are learned during training?

### The Intuition: The "Company They Keep"
Linguist J.R. Firth said in 1957: *"You shall know a word by the company it keeps."* 
If I say: "I spread the delicious ___ on my toast."
Even without knowing the word, you can guess it's "butter" or "jam". Word2Vec mathematically models this exact guessing game.

Word2Vec comes in two architectural flavors:

| Architecture | Objective | Intuition | Pros/Cons |
| :--- | :--- | :--- | :--- |
| **CBOW (Continuous Bag of Words)** | Predict the *target* word given the surrounding *context* words. | Like filling in the blank. "The cat sat on the [TARGET]" -> predict 'mat'. | Faster to train. Better for frequent words. |
| **Skip-Gram** | Predict the *context* words given a single *target* word. | Like brainstorming associations. Given 'mat', predict 'The', 'cat', 'sat'. | Slower. Vastly superior for infrequent/rare words. |

### The Math: Skip-Gram
Given a sequence of training words $w_{1}, w_{2}, \dots, w_{T}$, the goal is to maximize the average log probability of predicting context words given a center word:

$$ J(\theta) = \frac{1}{T} \sum_{t=1}^{T} \sum_{-c \le j \le c, j \neq 0} \log P(w_{t+j} | w_t) $$

Where $c$ is the window size. How do we define $P(w_{context} | w_{center})$?
We use two dense weight matrices. 
- $U$: the context word matrix (columns are vectors $u_o$)
- $V$: the center word matrix (columns are vectors $v_c$)

We use the **Softmax** function to convert dot products into probabilities:
$$ P(w_o | w_c) = \frac{\exp(u_o^T v_c)}{\sum_{w=1}^{|V|} \exp(u_w^T v_c)} $$

The numerator rewards the model if the context word $w_o$ vector points in the same direction as the center word $w_c$ vector. The denominator normalizes over the *entire vocabulary* to make it a valid probability distribution.


## 3. The Softmax Bottleneck & Negative Sampling

If you look closely at the Softmax denominator above, a massive problem emerges.

### The Intuition: The Grading Nightmare
Imagine a teacher grading a test. To score a student's answer (the numerator), they must also read and grade every other student's answer in the world (the denominator) to find the relative percentile. If the vocabulary size $|V|$ is 1,000,000 words, computing the denominator requires 1,000,000 dot products *for a single training step*. The model will take years to train.

### The Math: Negative Sampling
To solve this, Mikolov introduced **Negative Sampling**. Instead of doing a giant multi-class classification (predict 1 word out of $V$), we turn it into a series of **Binary Classifications**.

We ask: Is this pair of words a true context pair, or a fake one?
- True pair (from text): (apple, delicious) $\rightarrow$ Label 1
- Fake pairs (randomly drawn): (apple, spaceship), (apple, democracy) $\rightarrow$ Label 0

The new objective function becomes:
$$ \log \sigma(u_o^T v_c) + \sum_{i=1}^{k} \mathbb{E}_{w_i \sim P(w)} [\log \sigma(-u_i^T v_c)] $$

Where:
- $\sigma(x) = \frac{1}{1 + e^{-x}}$ (Sigmoid function)
- $w_o$ is the true context word.
- $w_i$ are $k$ "negative" words drawn randomly from a noise distribution $P(w)$.
- $k$ is usually between 5-20.

**Why this is genius:** We only update the weights for the target word, the true context word, and $k$ negative words. We bypass the massive denominator completely. Training becomes exponentially faster, allowing Word2Vec to scale to billions of words.

---

## 4. GloVe: Global Vectors & Matrix Factorization

Word2Vec is a predictive model. It only looks at local, rolling windows of text. Researchers at Stanford (Pennington et al., 2014) argued that predictive models miss the grand, overall statistics of the corpus. They introduced **GloVe**.

### The Intuition
Instead of playing a guessing game across local windows, what if we just count how many times every word appears with every other word in the entire Wikipedia dataset, build a massive spreadsheet (a co-occurrence matrix), and compress it?

If "ice" co-occurs frequently with "solid" but "steam" co-occurs frequently with "gas", the ratio of these counts inherently captures physical states. GloVe captures global semantic meaning directly from these global counts.

### The Math
1. Build a Co-occurrence Matrix $X$, where $X_{ij}$ is the number of times word $j$ appears in the context of word $i$.
2. GloVe posits that the dot product of two word embeddings should equal the log of their co-occurrence probability:

$$ w_i^T \tilde{w}_j + b_i + \tilde{b}_j = \log(X_{ij}) $$

GloVe trains embeddings by minimizing the squared error of this equation across the matrix, weighted by a function $f(X_{ij})$ that prevents extremely common word pairs (like "the", "and") from dominating the loss.

**Pros:** Leverages global statistics efficiently. Very good at analogies (King - Man + Woman = Queen).
**Cons:** Memory-heavy to build the massive co-occurrence matrix initially.

---

## 5. FastText: Subword Information & OOV Words

Word2Vec and GloVe still have a major weakness: **Out-of-Vocabulary (OOV) words**. If they haven't seen the exact word "tensor-tastic" during training, they crash. They also struggle with morphologically rich languages (like Turkish or German) where "spielen", "spielt", "gespielt" (play, plays, played) are treated as completely unrelated, independent vectors.

### The Intuition
Facebook AI introduced **FastText**. What if we don't just assign a vector to a whole word, but also to parts of the word (character n-grams)?

Take "apple". We pad it with brackets: `<apple>`
Break it into 3-grams: `<ap`, `app`, `ppl`, `ple`, `le>`

The final embedding for "apple" is the **sum** of its whole word vector plus the vectors for all its sub-n-grams.

### The Edge Case (Why this is powerful)
Suppose during deployment your model encounters a typo: "appple".
Word2Vec fails completely. FastText breaks it down: `<ap`, `app`, `ppp`, `ppl`, `ple`, `le>`.
It sums the subword vectors. Because many of these exact subwords were learned when training "apple" and "apples", the resulting vector for "appple" will be mathematically very close to "apple". It perfectly handles OOV words and misspellings!

---

## 6. Static vs. Contextual Embeddings (The Polysemy Problem)

Word2Vec, GloVe, and FastText are **Static Embeddings**.

### The Edge Case: Polysemy
Consider the sentences:
1. "I deposited money in the **bank**."
2. "I sat by the river **bank**."

In Word2Vec, the word "bank" has **exactly one vector representation**. It is essentially an average of the financial-bank meaning and river-bank meaning. When you pull the vector for "bank", you get a confused, blended mathematical state.

### Overview of the Solution: Contextual Embeddings
Models like **ELMo** and **BERT** revolutionized NLP by making embeddings *dynamic*. In BERT, there is no static dictionary where "bank" always equals `[0.1, 0.4, ...]`. Instead, you pass the *entire sentence* into the Transformer. The Attention mechanism analyzes the surrounding words and dynamically computes a unique embedding for "bank" on the fly.
- Sentence 1 "bank" vector $\approx$ meaning: financial institution.
- Sentence 2 "bank" vector $\approx$ meaning: geographical slope.

---

## 7. Sentence Transformers (SBERT) & Siamese Networks

If BERT can embed words contextually, can we just average the BERT word embeddings to get a single vector for an entire sentence? 

### The Intuition
Yes, but it's computationally horrific for semantic search. If you have 10,000 sentences and want to find the most similar one to your query, passing pairs through standard BERT requires $10,000$ massive neural network forward passes.

**Sentence-BERT (SBERT)** uses a **Siamese Network** architecture. You pass Sentence A through a BERT clone, and Sentence B through a tied-weights BERT clone. Both output a static fixed-size sentence embedding. You then calculate the Cosine Similarity between them.
By pre-computing the embeddings once and doing simple dot-products afterwards, search time drops from 65 hours to 50 milliseconds while retaining contextual richness.


## 8. PyTorch Implementation Demo (Skip-Gram)

Let's demystify embeddings by building a bare-bones Skip-Gram network from scratch. Notice how the embedding matrix is fundamentally just a lookup table that maps integer IDs to dense vectors.


In [2]:
torch.manual_seed(42)

# 1. Tiny Corpus
text = "the quick brown fox jumps over the lazy dog".split()
vocab = list(set(text))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
vocab_size = len(vocab)

# 2. Generate Skip-Gram pairs (Window Size = 1)
window_size = 1
data = []
for i in range(window_size, len(text) - window_size):
    target = word2idx[text[i]]
    context = [word2idx[text[i - window_size]], word2idx[text[i + window_size]]]
    for w in context:
        data.append((target, w))

print(f"Sample pairs (target, context): {data[:3]}")

# 3. The Model
class BareBonesSkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dims):
        super().__init__()
        # The Target Word embeddings (Matrix V)
        self.center_embeddings = nn.Embedding(vocab_size, embed_dims)
        # The Context Word embeddings (Matrix U)
        self.context_embeddings = nn.Embedding(vocab_size, embed_dims)
        
    def forward(self, target_idx, context_idx):
        # Extract specific row vectors based on IDs
        v_c = self.center_embeddings(target_idx) # Shape: (batch, embed_dims)
        u_o = self.context_embeddings(context_idx) # Shape: (batch, embed_dims)
        
        # Dot product to check alignment
        dot_product = torch.sum(v_c * u_o, dim=1)
        return dot_product

# Instantiation
EMBED_DIM = 2
model = BareBonesSkipGram(vocab_size, EMBED_DIM)
criterion = nn.BCEWithLogitsLoss() # Uses sigmoid under the hood to handle negative sampling binary logic
optimizer = optim.SGD(model.parameters(), lr=0.1)

# 4. Training Loop (Simulated Negative Sampling logic for illustrative purposes)
epochs = 1000
for epoch in range(epochs):
    total_loss = 0
    for target, true_context in data:
        # Positive pair
        t_pos = torch.tensor([target])
        c_pos = torch.tensor([true_context])
        y_pos = torch.tensor([1.0])
        
        # Random Negative pair (simulating 1 negative sample)
        fake_context = np.random.choice([i for i in range(vocab_size) if i != true_context])
        t_neg = torch.tensor([target])
        c_neg = torch.tensor([fake_context])
        y_neg = torch.tensor([0.0])
        
        # Forward passes
        pos_pred = model(t_pos, c_pos)
        neg_pred = model(t_neg, c_neg)
        
        loss = criterion(pos_pred, y_pos) + criterion(neg_pred, y_neg)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if (epoch+1) % 500 == 0:
        print(f"Epoch {epoch+1}/1000 | Loss: {total_loss:.4f}")


---

## 9. Master Interview Q&A Cheatsheet: Embeddings

> **Q: Fundamentally, why is negative sampling necessary in Word2Vec?**
> 
> **A:** Word2Vec relies on Softmax to predict the probability of a context word out of the entire vocabulary. The denominator of the Softmax requires calculating the dot product of the target word with *every single word in the vocabulary*. If the vocabulary is 1 million words, this is computationally implausible for every training instance. Negative Sampling bypasses this denominator bottleneck by transforming the problem from a giant multi-class classification into a series of binary classifications (Is this pair of words real or fake?), only updating the weights of the target word, the true context word, and $k$ random "negative" words.

> **Q: What is the main architectural difference between GloVe and Word2Vec?**
> 
> **A:** Word2Vec is a predictive, window-based model. It acts locally, sweeping a moving window across the text and playing a guessing game to update weights via backpropagation. GloVe (Global Vectors) is a count-based matrix factorization model. It acts globally, computing a massive vocabulary $\times$ vocabulary co-occurrence matrix containing the counts of how often words appear near each other across the entire corpus. It then factors this matrix to generate embeddings, ensuring that the dot product of two word vectors is directly proportional to the probability of those words co-occurring.

> **Q: Explain how FastText manages Out-Of-Vocabulary (OOV) words when traditional models fail.**
> 
> **A:** Traditional static models map a whole, exact string (like "apple") to a vector ID. If they encounter "appple" in production, the lookup fails. FastText maps embeddings to strings, but also generates embeddings for character sub-n-grams (e.g., `<ap`, `app`, `ppl`, `le>`). The final vocabulary vector is the sum of the whole word and its sub-n-grams. When an OOV word or typo appears, FastText simply extracts its sub-n-grams, queries those (which likely exist in the model), and sums them up, synthesizing a highly accurate semantic vector on the fly.

> **Q: Why are Sentence Transformers (SBERT) preferred over standard BERT for Semantic Search/Clustering tasks?**
> 
> **A:** Standard BERT requires passing a pair of sentences together through the Transformer cross-attention mechanisms to compute a similarity score. To find the most similar pair in a 10,000 sentence database, you must pass 49,995,000 pairs through a massive neural network, taking 65+ hours. SBERT utilizes a Siamese Architecture. It passes single sentences through tied-weight BERT blocks, outputting fixed-size static sentence embeddings. These embeddings can then be compared using simple, ultra-fast mathematical cosine similarity, reducing the 65 hour search down to ~50 milliseconds, making vector databases feasible.
